In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Per-model prompt caching across Gemini 4 Carbon, Neon, and Hydrogen

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/generative-ai/blob/main/gemini/context-caching/gemini_4_per_model_prompt_caching.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fgenerative-ai%2Fmain%2Fgemini%2Fcontext-caching%2Fgemini_4_per_model_prompt_caching.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/generative-ai/main/gemini/context-caching/gemini_4_per_model_prompt_caching.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/context-caching/gemini_4_per_model_prompt_caching.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/context-caching/gemini_4_per_model_prompt_caching.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/context-caching/gemini_4_per_model_prompt_caching.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/context-caching/gemini_4_per_model_prompt_caching.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/context-caching/gemini_4_per_model_prompt_caching.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/GoogleCloudPlatform/generative-ai/blob/main/gemini/context-caching/gemini_4_per_model_prompt_caching.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>

| Authors |
| --- |
| [Eric Dong](https://github.com/gericdong) |
| [Holt Skinner](https://github.com/holtskinner) |

## Overview

Gemini 4 Carbon, Neon, and Hydrogen use automatic **Implicit Context Caching** (95% discount on cached input tokens) once a shared prompt prefix reaches **4,096 tokens**. Explicit caching is not supported on Gemini 4.

| Model | Uncached Input (per 1M) | Cached Input (per 1M) | Output + Thinking (per 1M) |
| :--- | :--- | :--- | :--- |
| **Gemini 4 Hydrogen** (`gemini-4-hydrogen`) | \$1.50 | **\$0.075** | \$7.50 |
| **Gemini 4 Neon** (`gemini-4-neon`) | \$3.00 | **\$0.150** | \$15.00 |
| **Gemini 4 Carbon** (`gemini-4-carbon`, standard) | \$4.00 (\$2.00 promo through Nov 30) | **\$0.200** (\$0.100 promo) | \$20.00 (\$10.00 promo) |

Two things affect your cache hit rate in multi-model agents:
- **KV caches are per-model.** Handing off from Carbon or Neon to Hydrogen misses on Hydrogen's first turn (\$1.50/1M), then hits \$0.075/1M on consecutive Hydrogen turns. Meanwhile, the planner model's own cache stays warm when you switch back.
- **Prefixes match from token 0.** Putting a timestamp or turn counter at the top of `system_instruction` breaks prefix matching on every turn.

## Get started

### Install Google Gen AI SDK

> **Note:** You can safely ignore `pip` dependency resolver errors from unrelated pre-installed packages in Colab.

In [ ]:
%pip install --upgrade --quiet google-genai

### Authenticate your notebook environment

Authenticate your account if running in Google Colab.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Import libraries

In [ ]:
import os
import time
from typing import Any

from google import genai
from google.genai import types
from google.genai.errors import APIError

### Set Google Cloud project information

Set your Google Cloud project ID and initialize `genai.Client`.

In [ ]:
# fmt: off
PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on
LOCATION = "global"  # @param ["global", "us", "eu"]

if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "True"

client = genai.Client(
    enterprise=True,
    project=PROJECT_ID,
    location=LOCATION,
    http_options=types.HttpOptions(
        timeout=240000,
        retry_options=types.HttpRetryOptions(attempts=3),
    ),
)

### Model rates and shared prompt prefix

Set up a cost helper and a static prompt prefix above the 4,096-token caching threshold.

In [ ]:
CARBON_MODEL = "gemini-4-carbon"
NEON_MODEL = "gemini-4-neon"
HYDROGEN_MODEL = "gemini-4-hydrogen"

# Standard per-1M input token pricing: (uncached_usd, cached_usd)
INPUT_RATES_PER_1M: dict[str, tuple[float, float]] = {
    HYDROGEN_MODEL: (1.50, 0.075),
    NEON_MODEL: (3.00, 0.150),
    CARBON_MODEL: (4.00, 0.200),
}


def input_cost_per_1k_calls(
    model: str, prompt_tokens: int, cached_tokens: int
) -> float:
    uncached_rate, cached_rate = INPUT_RATES_PER_1M[model]
    uncached = max(0, prompt_tokens - cached_tokens)
    cost_one_call = (
        uncached * uncached_rate + cached_tokens * cached_rate
    ) / 1_000_000
    return cost_one_call * 1_000


SHARED_REPO_PREFIX = (
    "You are a distributed systems coding agent. Answer concisely in 1 sentence.\n\n"
    + "\n".join(
        f"## Service Module svc-{i:03d}\n"
        f"- Owner: platform-core | Region: us-east1 | Protocol: gRPC + Protobuf\n"
        f"- Retry Policy: exponential_jitter(base_ms=80, cap_ms={600 + i})\n"
        f"- Idempotency: required header `X-Idempotency-Key` with 24h Redis TTL."
        for i in range(1, 185)
    )
)

## Explicit caching check

If you are migrating from Gemini 3.x, remove `caches.create` calls and `cached_content` fields. Gemini 4 rejects explicit cache creation with `HTTP 400`.

In [ ]:
try:
    client.caches.create(
        model=NEON_MODEL,
        config=types.CreateCachedContentConfig(
            contents=[SHARED_REPO_PREFIX],
            ttl="300s",
        ),
    )
except APIError as err:
    print(f"Expected guardrail -> HTTP {err.code}: {err.message[:85]}")

Expected guardrail -> HTTP 400: Model `gemini-4-neon` does not support explicit context caching.


## Prefix ordering: dynamic-first vs. static-first

Implicit caching matches tokens from the start of the request (`system_instruction` + `tools` + `contents`). Putting changing text at the start of `system_instruction` results in 0 cached tokens.

### Dynamic header at the start (misses cache)

In [ ]:
anti_pattern_prefix = SHARED_REPO_PREFIX.replace(
    "distributed systems", "anti-pattern demo"
)

for turn in (1, 2):
    # Bad: dynamic header at index 0 changes every turn and breaks prefix matching
    bad_sys = (
        f"[Turn={turn} | RequestTimestamp={1763500000 + turn}]\n"
        + anti_pattern_prefix
    )
    resp = client.models.generate_content(
        model=HYDROGEN_MODEL,
        contents=f"What is cap_ms for svc-0{turn}0?",
        config=types.GenerateContentConfig(
            system_instruction=bad_sys,
            thinking_config=types.ThinkingConfig(thinking_level="LOW"),
        ),
    )
    u = resp.usage_metadata
    print(
        f"Dynamic-first Turn {turn}: prompt={u.prompt_token_count}, "
        f"cached={u.cached_content_token_count or 0}"
    )

Dynamic-first Turn 1: prompt=14805, cached=0
Dynamic-first Turn 2: prompt=14805, cached=0


### Static instructions first, dynamic text at the end (hits cache)

In [ ]:
static_first_prefix = SHARED_REPO_PREFIX.replace(
    "distributed systems", "static-first demo"
)

for turn in (1, 2, 3):
    resp = client.models.generate_content(
        model=HYDROGEN_MODEL,
        contents=f"[Turn={turn}] What is cap_ms for svc-0{turn}0?",
        config=types.GenerateContentConfig(
            system_instruction=static_first_prefix,
            thinking_config=types.ThinkingConfig(thinking_level="LOW"),
        ),
    )
    u = resp.usage_metadata
    cached = u.cached_content_token_count or 0
    hit_pct = 100.0 * cached / u.prompt_token_count
    print(
        f"Static-first Turn {turn}: prompt={u.prompt_token_count}, "
        f"cached={cached} ({hit_pct:4.1f}%)"
    )

Static-first Turn 1: prompt=14790, cached=0 ( 0.0%)
Static-first Turn 2: prompt=14790, cached=12736 (86.1%)
Static-first Turn 3: prompt=14790, cached=12741 (86.1%)


## Per-model KV cache isolation across Carbon, Neon, and Hydrogen

Run a 6-step sequence over the same shared prefix (`Carbon -> Neon -> Hydrogen -> Hydrogen -> Neon -> Carbon`):
- Steps 1 to 3 each use a model for the first time, so `cached` is `0` on all three.
- Step 4 stays on Hydrogen and hits Hydrogen's cache.
- Steps 5 and 6 switch back to Neon and Carbon and hit each model's still-warm cache.

In [ ]:
isolation_prefix = SHARED_REPO_PREFIX.replace(
    "distributed systems", "cross-model isolation demo"
)

handoff_schedule = [
    (CARBON_MODEL, "Step 1 (Carbon cold prefill): check cap_ms for svc-005."),
    (NEON_MODEL, "Step 2 (Neon cold prefill): check cap_ms for svc-015."),
    (HYDROGEN_MODEL, "Step 3 (Hydrogen cold prefill): check cap_ms for svc-025."),
    (HYDROGEN_MODEL, "Step 4 (Hydrogen consecutive): check cap_ms for svc-035."),
    (NEON_MODEL, "Step 5 (Hand back to warm Neon): check cap_ms for svc-045."),
    (CARBON_MODEL, "Step 6 (Hand back to warm Carbon): check cap_ms for svc-055."),
]

history: list[types.Content] = []
print(
    f"{'Step & Model':<28} | {'Prompt':>7} | {'Cached':>7} | "
    f"{'Hit %':>6} | {'$/1K Calls':>10}"
)
print("-" * 70)

for idx, (model_id, user_msg) in enumerate(handoff_schedule, start=1):
    history.append(
        types.Content(role="user", parts=[types.Part.from_text(text=user_msg)])
    )
    resp = client.models.generate_content(
        model=model_id,
        contents=history,
        config=types.GenerateContentConfig(
            system_instruction=isolation_prefix,
            thinking_config=types.ThinkingConfig(thinking_level="LOW"),
        ),
    )
    history.append(resp.candidates[0].content)
    u = resp.usage_metadata
    prompt_tok = u.prompt_token_count or 0
    cached_tok = u.cached_content_token_count or 0
    hit_pct = (100.0 * cached_tok / prompt_tok) if prompt_tok else 0.0
    cost_1k = input_cost_per_1k_calls(model_id, prompt_tok, cached_tok)
    label = f"{idx}. {model_id}"
    print(
        f"{label:<28} | {prompt_tok:>7} | {cached_tok:>7} | "
        f"{hit_pct:>5.1f}% | ${cost_1k:>9.3f}"
    )

Step & Model                 |  Prompt |  Cached |  Hit % | $/1K Calls
----------------------------------------------------------------------
1. gemini-4-carbon           |   14794 |       0 |   0.0% | $   59.176
2. gemini-4-neon             |   14848 |       0 |   0.0% | $   44.544
3. gemini-4-hydrogen         |   14894 |       0 |   0.0% | $   22.341
4. gemini-4-hydrogen         |   15026 |   12845 |  85.5% | $    4.235
5. gemini-4-neon             |   15169 |   12799 |  84.4% | $    9.030
6. gemini-4-carbon           |   15224 |   12740 |  83.7% | $   12.484


## 12-turn agent cost comparison

Compare a 12-turn coding loop (1 plan turn, 10 worker turns, 1 review turn; 15K starting input tokens, +500 input and 1,200 thinking/output tokens per turn) across four setups.

In [ ]:
OUTPUT_RATES_PER_1M: dict[str, float] = {
    HYDROGEN_MODEL: 7.50,
    NEON_MODEL: 15.00,
    CARBON_MODEL: 20.00,
}


def simulate_12_turn_tco(
    schedule: list[tuple[str, bool]],
    base_in: int = 15_000,
    turn_growth: int = 500,
    out_per_turn: int = 1_200,
) -> float:
    seen: set[str] = set()
    total_per_1k = 0.0
    for idx, (mdl, stable) in enumerate(schedule):
        in_tok = base_in + idx * turn_growth
        cached_tok = (
            (base_in + max(0, idx - 1) * turn_growth)
            if (stable and mdl in seen)
            else 0
        )
        if stable:
            seen.add(mdl)
        in_cost = input_cost_per_1k_calls(mdl, in_tok, cached_tok)
        out_cost = (out_per_turn * OUTPUT_RATES_PER_1M[mdl] / 1_000_000) * 1_000
        total_per_1k += in_cost + out_cost
    return total_per_1k


schedules = {
    "A. All Carbon (broken prefix, 0% cache)": [(CARBON_MODEL, False)] * 12,
    "B. All Carbon (stable prefix, 95% cache)": [(CARBON_MODEL, True)] * 12,
    "C. All Neon (stable prefix, 95% cache)": [(NEON_MODEL, True)] * 12,
    "D. Neon plan/review + 10 batched Hydrogen": (
        [(NEON_MODEL, True)]
        + [(HYDROGEN_MODEL, True)] * 10
        + [(NEON_MODEL, True)]
    ),
}

for label, sched in schedules.items():
    tco = simulate_12_turn_tco(sched)
    print(f"{label:<43} -> ${tco:>7.2f} total / 1,000 agent runs")

A. All Carbon (broken prefix, 0% cache)     -> $1140.00 total / 1,000 agent runs
B. All Carbon (stable prefix, 95% cache)    -> $ 408.50 total / 1,000 agent runs
C. All Neon (stable prefix, 95% cache)      -> $ 306.38 total / 1,000 agent runs
D. Neon plan/review + 10 batched Hydrogen   -> $ 217.31 total / 1,000 agent runs


## Batched handoffs on the Interactions API

Chaining turns with `previous_interaction_id` (`store=True`) follows the same per-model cache behavior. Keeping `system_instruction` fixed across consecutive `gemini-4-hydrogen` worker turns hits `interaction.usage.total_cached_tokens` on Turn 2+ (implicit caching is best-effort across serving replicas, so grouping worker turns on one model gives the highest hit rate).

In [ ]:
interactions_prefix = SHARED_REPO_PREFIX.replace(
    "distributed systems", "interactions-api cache demo"
)
stored_ids: list[str] = []

turns_spec = [
    (NEON_MODEL, "low", "Plan: list svc-001 and svc-002 cap_ms values."),
    (HYDROGEN_MODEL, "low", "Worker 1: compute their sum."),
    (HYDROGEN_MODEL, "low", "Worker 2: compute their difference."),
    (HYDROGEN_MODEL, "low", "Worker 3: format both numbers as JSON."),
    (NEON_MODEL, "low", "Verify: confirm the JSON matches svc-001 and svc-002."),
]

prev_id: str | None = None
print(f"{'Turn & Model':<28} | {'Input Tok':>9} | {'Cached Tok':>10} | {'Hit %':>6}")
print("-" * 62)

for idx, (mdl, level, prompt) in enumerate(turns_spec, start=1):
    kwargs: dict[str, Any] = {
        "model": mdl,
        "system_instruction": interactions_prefix,
        "input": prompt,
        "generation_config": {"thinking_level": lvl},
        "store": True,
        "timeout": 180,
    }
    if prev_id is not None:
        kwargs["previous_interaction_id"] = prev_id

    inter = client.interactions.create(**kwargs)
    stored_ids.append(inter.id)
    prev_id = inter.id

    in_tok = inter.usage.total_input_tokens or 0
    cached_tok = inter.usage.total_cached_tokens or 0
    pct = (100.0 * cached_tok / in_tok) if in_tok else 0.0
    print(f"{f'{idx}. {mdl}':<28} | {in_tok:>9} | {cached_tok:>10} | {pct:>5.1f}%")

Turn & Model                 | Input Tok | Cached Tok |  Hit %
--------------------------------------------------------------
1. gemini-4-neon             |     14793 |          0 |   0.0%
2. gemini-4-hydrogen         |     14844 |          0 |   0.0%
3. gemini-4-hydrogen         |     14907 |      12795 |  85.8%
4. gemini-4-hydrogen         |     14969 |      12858 |  85.9%
5. gemini-4-neon             |     15171 |          0 |   0.0%


## Cleaning up

Delete the stored interactions from the previous section (stored interactions otherwise expire after 7 days).

In [ ]:
for interaction_id in stored_ids:
    client.interactions.delete(interaction_id)

print(f"Deleted {len(stored_ids)} stored interactions.")

Deleted 5 stored interactions.
